In [15]:
import os  # Provides functions for working with the operating system (files, directories, paths)
import numpy as np   # Used for numerical operations and handling arrays
import pandas as pd  # Used for data manipulation and analysis in tabular form
import librosa   # A library for audio and music processing (feature extraction, loading sounds)
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical


In [50]:
dataset_path = r"D:\TalentPrism\Voice Detection\archive\data"  # Define the root path of the dataset containing different labeled folders

# Initialize two empty lists to store file paths and their corresponding labels
paths = []
labels = []

# Loop through each folder (label) inside the dataset directory
for label in os.listdir(dataset_path):
    label_folder = os.path.join(dataset_path, label)
    if os.path.isdir(label_folder):
        for file in os.listdir(label_folder):
            if file.endswith(".wav"):
                paths.append(os.path.join(label_folder, file))
                labels.append(label.lower())

print(f"Total samples: {len(paths)}")
print(f"Labels: {set(labels)}")    # Print the unique set of labels found in the dataset

Total samples: 16148
Labels: {'male', 'female'}


### Why MFCC?

They reduce high-dimensional audio data into a smaller feature vector while preserving information useful for distinguishing different sounds

In [66]:
def extract_mfcc(file_path, n_mfcc=40):  # to extract MFCC (Mel-Frequency Cepstral Coefficients) features from an audio file
    try:
        audio, sr = librosa.load(file_path, sr=None)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        mfcc_mean = np.mean(mfcc.T, axis=0)
        return mfcc_mean
    except Exception as e:
        print("Error:", file_path, e)
        return None

# to hold feature vectors (X) and their corresponding labels (y)
X = []   
y = []

for path, label in zip(paths, labels):
    mfcc = extract_mfcc(path)
    if mfcc is not None:
        X.append(mfcc)
        y.append(label)

X = np.array(X)
y = np.array(y)
print("X shape:", X.shape, "y shape:", y.shape)

X shape: (16148, 40) y shape: (16148,)


In [67]:
# Create a LabelEncoder object to convert string labels into numeric labels
le = LabelEncoder()
# Fit the encoder on y (learn all unique classes) and transform them into integers
y = le.fit_transform(y)  # male=1, female=0
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Label mapping: {'female': 0, 'male': 1}


In [70]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [71]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [72]:
# Build a Sequential neural network model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model: define optimizer, loss function, and evaluation metrics
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display a summary of the model architecture (layers, shapes, parameters)
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_8 (Dense)             (None, 64)                2624      
                                                                 
 dropout_5 (Dropout)         (None, 64)                0         
                                                                 
 dense_9 (Dense)             (None, 32)                2080      
                                                                 
 dropout_6 (Dropout)         (None, 32)                0         
                                                                 
 dense_10 (Dense)            (None, 1)                 33        
                                                                 
Total params: 4737 (18.50 KB)
Trainable params: 4737 (18.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [73]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Compute class weights to handle imbalance
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
# Create a dictionary mapping: {class_label: weight}
class_weights = dict(zip(classes, weights))
print("Class weights:", class_weights)


Class weights: {0: 1.3998699609882965, 1: 0.7778179190751445}


In [57]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,    # Number of passes over the training data
    batch_size=32,
    shuffle=True
    class_weight=class_weights  # Apply class weights to handle class imbalance
)


Epoch 1/30
323/323 [==============================] - 3s 5ms/step - loss: 0.2218 - accuracy: 0.9136 - val_loss: 0.0580 - val_accuracy: 0.9830
Epoch 2/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0701 - accuracy: 0.9758 - val_loss: 0.0437 - val_accuracy: 0.9861
Epoch 3/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0506 - accuracy: 0.9840 - val_loss: 0.0384 - val_accuracy: 0.9884
Epoch 4/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0433 - accuracy: 0.9864 - val_loss: 0.0356 - val_accuracy: 0.9892
Epoch 5/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0404 - accuracy: 0.9864 - val_loss: 0.0302 - val_accuracy: 0.9911
Epoch 6/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0370 - accuracy: 0.9881 - val_loss: 0.0323 - val_accuracy: 0.9911
Epoch 7/30
323/323 [==============================] - 1s 4ms/step - loss: 0.0302 - accuracy: 0.9906 - val_loss: 0.0304 - val_accuracy: 0.9911
Epoch 

In [74]:
# Evaluate the trained model on unseen test data
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

101/101 [==============================] - 1s 3ms/step - loss: 0.6133 - accuracy: 0.6845
Test Accuracy: 68.45%


# Prediction of Gender

In [75]:
def predict_gender(file_path):
    # Extract MFCC features from the new audio file (same process as training)
    mfcc = extract_mfcc(file_path)
    if mfcc is None:
        # Handle case where audio processing failed
        return "Error processing audio"
    mfcc = scaler.transform([mfcc])
    prob = model.predict(mfcc, verbose=0)[0][0]
    label_index = int(prob >= 0.5)
    label = le.inverse_transform([label_index])[0]
    print("Probability:", prob)
    return label

In [78]:
print(predict_gender(r"D:\TalentPrism\Voice Detection\archive\testaudio\test.wav"))

Probability: 0.4137443
female
